성별 x 쿠폰타입 전환율
연령대 x 쿠폰타입 평균 결제액
소득 구간 x 쿠폰타입 달성률
멤버십 가입 연도 x 쿠폰타입 반응률 (오래된 고객 vs 신규 고객)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display
import warnings

import ast

warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'

# 마이너스 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

# 전역 시드 설정 (재현성을 위해)
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [2]:
portfolio = pd.read_csv('../1data/portfolio.csv')
profile = pd.read_csv('../1data/profile.csv')
transcript = pd.read_csv('../1data/transcript.csv')

merged = pd.read_csv('../1data/all_merged.csv')
promotion = pd.read_csv('../1data/promotion_df.csv')
normal = pd.read_csv('../1data/normal_flow_df.csv')

---

In [14]:
promotion.head()

,person,offer_id,offer_cycle,offer received,offer viewed,offer completed,transcript_reward,offer_type,difficulty,reward,...,mobile,social,gender,age,became_member_on,income,amount,is_received,is_viewed,is_completed
0,0009655768c64bdeb2e877511632db8f,bogo_5_5_5,Bogo_1,408.0,456.0,414.0,5.0,bogo,5,5,...,1,1,M,33,2017-04-21,72000.0,8.57,1,1,1
1,0009655768c64bdeb2e877511632db8f,discount_10_2_10,Discount_1,504.0,540.0,528.0,2.0,discount,10,2,...,1,1,M,33,2017-04-21,72000.0,14.11,1,1,1
2,0009655768c64bdeb2e877511632db8f,discount_10_2_7,Discount_1,576.0,NaN,576.0,2.0,discount,10,2,...,1,0,M,33,2017-04-21,72000.0,10.27,1,0,1
3,0009655768c64bdeb2e877511632db8f,informational_0_0_3,Informational_1,168.0,192.0,NaN,NaN,informational,0,0,...,1,1,M,33,2017-04-21,72000.0,NaN,1,1,0
4,0009655768c64bdeb2e877511632db8f,informational_0_0_4,Informational_1,336.0,372.0,NaN,NaN,informational,0,0,...,1,0,M,33,2017-04-21,72000.0,NaN,1,1,0


---

# 고객 전체 흐름

## 1. 성별x구폰타입 전환율

In [4]:
# bogo, discount만 필터링
bd_df = promotion[promotion['offer_type'].isin(['bogo', 'discount'])].copy()

# 성별 x offer_type 퍼널 전환율
gender_funnel = (
    bd_df
    .groupby(['gender', 'offer_type'])
    .agg(
        received=('is_received', 'sum'),
        viewed=('is_viewed', 'sum'),
        completed=('is_completed', 'sum')
    )
    .reset_index()
)
gender_funnel['viewed_rate'] = (gender_funnel['viewed'] / gender_funnel['received'] * 100).round(2)
gender_funnel['completed_rate'] = (gender_funnel['completed'] / gender_funnel['viewed'] * 100).round(2)
gender_funnel['total_rate'] = (gender_funnel['completed'] / gender_funnel['received'] * 100).round(2)
display(gender_funnel)

# 성별 x offer_id 퍼널 전환율
gender_funnel_id = (
    bd_df
    .groupby(['gender', 'offer_id'])
    .agg(
        received=('is_received', 'sum'),
        viewed=('is_viewed', 'sum'),
        completed=('is_completed', 'sum')
    )
    .reset_index()
)
gender_funnel_id['viewed_rate'] = (gender_funnel_id['viewed'] / gender_funnel_id['received'] * 100).round(2)
gender_funnel_id['completed_rate'] = (gender_funnel_id['completed'] / gender_funnel_id['viewed'] * 100).round(2)
gender_funnel_id['total_rate'] = (gender_funnel_id['completed'] / gender_funnel_id['received'] * 100).round(2)
display(gender_funnel_id)

,gender,offer_type,received,viewed,completed,viewed_rate,completed_rate,total_rate
0,F,bogo,11042,9198,7501,83.30,81.55,67.93
1,F,discount,11056,7806,7976,70.60,102.18,72.14
2,M,bogo,15295,12652,7512,82.72,59.37,49.11
3,M,discount,15523,10559,8954,68.02,84.80,57.68
4,O,bogo,358,319,245,89.11,76.80,68.44
5,O,discount,371,300,256,80.86,85.33,69.00


,gender,offer_id,received,viewed,completed,viewed_rate,completed_rate,total_rate
0,F,bogo_10_10_5,2752,2637,1746,95.82,66.21,63.44
1,F,bogo_10_10_7,2773,2384,1857,85.97,77.89,66.97
2,F,bogo_5_5_5,2730,2621,1899,96.01,72.45,69.56
3,F,bogo_5_5_7,2787,1556,1999,55.83,128.47,71.73
4,F,discount_10_2_10,2719,2631,2216,96.76,84.23,81.50
5,F,discount_10_2_7,2750,1544,1851,56.15,119.88,67.31
6,F,discount_20_5_10,2852,1000,1703,35.06,170.30,59.71
7,F,discount_7_3_7,2735,2631,2206,96.20,83.85,80.66
8,M,bogo_10_10_5,3798,3648,1519,96.05,41.64,39.99
9,M,bogo_10_10_7,3869,3480,1741,89.95,50.03,45.00


- 여성이 남성보다 전환율이 높음
- `discount`가 `bogo`보다 전환율이 높음
- 여성에게 가장 효과적인 쿠폰은 `discount_10_2_10`(81.50%)과 `discount_7_3_7`(80.66%)
- 남성에서도 `discount_10_2_10`(81.50%)과 `discount_7_3_7`(80.66%)가 상위권임
- `discount_20_5_10`는 남녀 모두 다른 `discount` 쿠폰 대비 전환율이 낮음

20$ -> bogo, discount 둘 다 사용됨 <br>
쿠폰별 => bogo, discount completed -> 더블 카운팅 제거 -> bogo completed -> disc 누락 <br>
비용면 => 더블 카운팅 제거x -> 총합이 2개가 체크 -> 제거를 해야하고

## 2. 연령대 x 쿠폰타입 평균 결제액

In [5]:
bd_df['age_group'] = pd.cut(bd_df['age'], bins=[0, 20, 30, 40, 50, 60, 70, 100], labels=['10대', '20대', '30대', '40대', '50대', '60대', '70대+'])

# 연령대 x offer_type 퍼널 전환율
age_funnel = (
    bd_df
    .groupby(['age_group', 'offer_type'])
    .agg(
        received=('is_received', 'sum'),
        viewed=('is_viewed', 'sum'),
        completed=('is_completed', 'sum')
    )
    .reset_index()
)
age_funnel['viewed_rate'] = (age_funnel['viewed'] / age_funnel['received'] * 100).round(2)
age_funnel['completed_rate'] = (age_funnel['completed'] / age_funnel['viewed'] * 100).round(2)
age_funnel['total_rate'] = (age_funnel['completed'] / age_funnel['received'] * 100).round(2)
display(age_funnel)

# 연령대 x offer_id 퍼널 전환율
age_funnel_id = (
    bd_df
    .groupby(['age_group', 'offer_id'])
    .agg(
        received=('is_received', 'sum'),
        viewed=('is_viewed', 'sum'),
        completed=('is_completed', 'sum')
    )
    .reset_index()
)
age_funnel_id['viewed_rate'] = (age_funnel_id['viewed'] / age_funnel_id['received'] * 100).round(2)
age_funnel_id['completed_rate'] = (age_funnel_id['completed'] / age_funnel_id['viewed'] * 100).round(2)
age_funnel_id['total_rate'] = (age_funnel_id['completed'] / age_funnel_id['received'] * 100).round(2)
display(age_funnel_id)

,age_group,offer_type,received,viewed,completed,viewed_rate,completed_rate,total_rate
0,10대,bogo,640,524,244,81.88,46.56,38.12
1,10대,discount,568,335,292,58.98,87.16,51.41
2,20대,bogo,2451,1994,1066,81.35,53.46,43.49
3,20대,discount,2574,1543,1344,59.95,87.10,52.21
4,30대,bogo,2850,2353,1439,82.56,61.16,50.49
5,30대,discount,2975,1980,1754,66.55,88.59,58.96
6,40대,bogo,4337,3714,2448,85.64,65.91,56.44
7,40대,discount,4239,3088,2658,72.85,86.08,62.70
8,50대,bogo,6454,5336,3991,82.68,74.79,61.84
9,50대,discount,6553,4596,4366,70.14,95.00,66.63


,age_group,offer_id,received,viewed,completed,viewed_rate,completed_rate,total_rate
0,10대,bogo_10_10_5,166,161,44,96.99,27.33,26.51
1,10대,bogo_10_10_7,158,150,54,94.94,36.00,34.18
2,10대,bogo_5_5_5,169,162,79,95.86,48.77,46.75
3,10대,bogo_5_5_7,147,51,67,34.69,131.37,45.58
4,10대,discount_10_2_10,127,122,79,96.06,64.75,62.20
5,10대,discount_10_2_7,153,46,63,30.07,136.96,41.18
6,10대,discount_20_5_10,132,22,45,16.67,204.55,34.09
7,10대,discount_7_3_7,156,145,105,92.95,72.41,67.31
8,20대,bogo_10_10_5,640,602,195,94.06,32.39,30.47
9,20대,bogo_10_10_7,598,579,204,96.82,35.23,34.11


- 나이가 많을수록 평균 결제액이 높아짐
    - 10대는 약 12~13달러 / 50~70대는 22~24달러로 약 2배 차이
- 10~30대는 `bogo`가 70대+에서는 `discount`의 평균 결제액이 높으며 40~60대는 두개가 비슷
- `discount_20_5_10`이 전 연령대에서 평균 결제액이 가장 높음
    - 난이도가 높은 만큼 달성한 고객들의 실제 결제액도 높아지는 듯 함
- `bogo_10_10` 시리즈 도 전 연령대에서 상위권을 차지
- 고연령층 타켓에는 `discount_20_5_10`이 객단가를 높이는 데 가장 효과적이며 저연령층은 난이도가 낮은 쿠폰 위주로 접근하면 전환율 측면에서 유리할 듯 함

## 3. 소득 구간 x 쿠폰타입 달성률

In [6]:
bd_df['income_group'] = pd.cut(bd_df['income'],
    bins=[0, 40000, 60000, 80000, 100000, 120000],
    labels=['4만이하', '4~6만', '6~8만', '8~10만', '10만이상'])

# 소득 구간 x offer_type 퍼널 전환율
income_funnel = (
    bd_df
    .groupby(['income_group', 'offer_type'])
    .agg(
        received=('is_received', 'sum'),
        viewed=('is_viewed', 'sum'),
        completed=('is_completed', 'sum')
    )
    .reset_index()
)
income_funnel['viewed_rate'] = (income_funnel['viewed'] / income_funnel['received'] * 100).round(2)
income_funnel['completed_rate'] = (income_funnel['completed'] / income_funnel['viewed'] * 100).round(2)
income_funnel['total_rate'] = (income_funnel['completed'] / income_funnel['received'] * 100).round(2)
display(income_funnel)

# 소득 구간 x offer_id 퍼널 전환율
income_funnel_id = (
    bd_df
    .groupby(['income_group', 'offer_id'])
    .agg(
        received=('is_received', 'sum'),
        viewed=('is_viewed', 'sum'),
        completed=('is_completed', 'sum')
    )
    .reset_index()
)
income_funnel_id['viewed_rate'] = (income_funnel_id['viewed'] / income_funnel_id['received'] * 100).round(2)
income_funnel_id['completed_rate'] = (income_funnel_id['completed'] / income_funnel_id['viewed'] * 100).round(2)
income_funnel_id['total_rate'] = (income_funnel_id['completed'] / income_funnel_id['received'] * 100).round(2)
display(income_funnel_id)

,income_group,offer_type,received,viewed,completed,viewed_rate,completed_rate,total_rate
0,4만이하,bogo,3926,3140,1485,79.98,47.29,37.82
1,4만이하,discount,3917,2326,1893,59.38,81.38,48.33
2,4~6만,bogo,8147,6802,3995,83.49,58.73,49.04
3,4~6만,discount,8276,5605,4723,67.73,84.26,57.07
4,6~8만,bogo,8131,6913,4978,85.02,72.01,61.22
5,6~8만,discount,8344,6009,5569,72.02,92.68,66.74
6,8~10만,bogo,4647,3921,3423,84.38,87.30,73.66
7,8~10만,discount,4581,3596,3555,78.50,98.86,77.60
8,10만이상,bogo,1844,1393,1377,75.54,98.85,74.67
9,10만이상,discount,1832,1129,1446,61.63,128.08,78.93


,income_group,offer_id,received,viewed,completed,viewed_rate,completed_rate,total_rate
0,4만이하,bogo_10_10_5,947,897,242,94.72,26.98,25.55
1,4만이하,bogo_10_10_7,980,938,272,95.71,29.00,27.76
2,4만이하,bogo_5_5_5,951,899,461,94.53,51.28,48.48
3,4만이하,bogo_5_5_7,1048,406,510,38.74,125.62,48.66
4,4만이하,discount_10_2_10,978,931,609,95.19,65.41,62.27
5,4만이하,discount_10_2_7,956,314,367,32.85,116.88,38.39
6,4만이하,discount_20_5_10,1011,163,280,16.12,171.78,27.70
7,4만이하,discount_7_3_7,972,918,637,94.44,69.39,65.53
8,4~6만,bogo_10_10_5,2030,1952,800,96.16,40.98,39.41
9,4~6만,bogo_10_10_7,2045,1907,902,93.25,47.30,44.11


- 소득이 높을수록 전환율이 올라감
    - 4만 이하보다 10만 이상은 약 2배 차이가 남
- 전 소득 구간에서 `bogo`보다 `discount` 전환율이 높음
- 4만 이하에서는 `bogo_10_10` 시리즈의 전환율이 낮지만 `discount_10_2_10`(62.27%)과 `discount_7_3_7`(65.53%)은 상대적으로 높음
- 10만 이상에서는 `discount_10_2_10`(89.74%)과 `discount_7_3_7`(82.39%)이 높은 전환율을 보임
- 6~8만에서도 `discount_10_2_10`(76.87%)과 `discount_7_3_7`(74.75%)이 상위권
- `discount_20_5_10`은 난이도가 높아 저소득층에서 전환율이 매우 낮고, 소득이 올라갈수록 전환율도 올라감
- 소득 구간별 타겟 쿠폰을 다르게 설계하는 것이 효과적일듯
    - 저소득층에는 난이도가 낮은 쿠폰 위주로
    - 고소득층에는 난이도가 높은 쿠폰 위주로

## 4. 멤버십 가입 연도 x 쿠폰타입 반응률 (오래된 고객 vs 신규 고객)

In [7]:
bd_df['join_year'] = pd.to_datetime(bd_df['became_member_on']).dt.year

# 가입 연도 x offer_type 퍼널 전환율
join_funnel = (
    bd_df
    .groupby(['join_year', 'offer_type'])
    .agg(
        received=('is_received', 'sum'),
        viewed=('is_viewed', 'sum'),
        completed=('is_completed', 'sum')
    )
    .reset_index()
)
join_funnel['viewed_rate'] = (join_funnel['viewed'] / join_funnel['received'] * 100).round(2)
join_funnel['completed_rate'] = (join_funnel['completed'] / join_funnel['viewed'] * 100).round(2)
join_funnel['total_rate'] = (join_funnel['completed'] / join_funnel['received'] * 100).round(2)
display(join_funnel)

# 가입 연도 x offer_id 퍼널 전환율
join_funnel_id = (
    bd_df
    .groupby(['join_year', 'offer_id'])
    .agg(
        received=('is_received', 'sum'),
        viewed=('is_viewed', 'sum'),
        completed=('is_completed', 'sum')
    )
    .reset_index()
)
join_funnel_id['viewed_rate'] = (join_funnel_id['viewed'] / join_funnel_id['received'] * 100).round(2)
join_funnel_id['completed_rate'] = (join_funnel_id['completed'] / join_funnel_id['viewed'] * 100).round(2)
join_funnel_id['total_rate'] = (join_funnel_id['completed'] / join_funnel_id['received'] * 100).round(2)
display(join_funnel_id)

,join_year,offer_type,received,viewed,completed,viewed_rate,completed_rate,total_rate
0,2013,bogo,528,451,229,85.42,50.78,43.37
1,2013,discount,509,343,339,67.39,98.83,66.60
2,2014,bogo,1230,1029,501,83.66,48.69,40.73
3,2014,discount,1265,899,833,71.07,92.66,65.85
4,2015,bogo,3339,2805,2092,84.01,74.58,62.65
5,2015,discount,3323,2285,2397,68.76,104.90,72.13
6,2016,bogo,6302,5271,4385,83.64,83.19,69.58
7,2016,discount,6388,4597,4913,71.96,106.87,76.91
8,2017,bogo,11735,9736,6014,82.97,61.77,51.25
9,2017,discount,11609,8155,6581,70.25,80.70,56.69


,join_year,offer_id,received,viewed,completed,viewed_rate,completed_rate,total_rate
0,2013,bogo_10_10_5,126,122,40,96.83,32.79,31.75
1,2013,bogo_10_10_7,140,125,44,89.29,35.20,31.43
2,2013,bogo_5_5_5,121,119,68,98.35,57.14,56.20
3,2013,bogo_5_5_7,141,85,77,60.28,90.59,54.61
4,2013,discount_10_2_10,128,121,116,94.53,95.87,90.62
5,2013,discount_10_2_7,125,70,72,56.00,102.86,57.60
6,2013,discount_20_5_10,130,32,43,24.62,134.38,33.08
7,2013,discount_7_3_7,126,120,108,95.24,90.00,85.71
8,2014,bogo_10_10_5,330,311,92,94.24,29.58,27.88
9,2014,bogo_10_10_7,304,269,88,88.49,32.71,28.95


- 2013~2016년 가입 고객은 전환율이 높고, 2017~2018년 가입 고객은 전환율이 낮음
- 전 연도에서 `discount`가 `bogo`보다 높음
- 2013~2014년 가입자가 전환율이 낮은데, 초기 멥버십 가입자 특성이 달라서 일 수도?
- `discount_10_2_10`이 전 연도에서 가장 높은 전환율을 보임
- `discount_7_3_7`도 전반적으로 높은 전환율을 보임
- 반면 `discount_20_5_10`은 낮은 전환율을 보임

---

위에랑 같이 모아서 보면 좋겠다

정상 흐름 보기 / 전체 받기 -> 정상흐름의 전환율
전체 보기 / 정상 흐름 완료 -> --

# 고객  정상 흐름

## 1. 성별x구폰타입 전환율

In [ ]:
# bogo, discount만 필터링
bd_df2 = promotion[promotion['offer_type'].isin(['bogo', 'discount'])].copy()

# 정상 흐름 여부 컬럼 추가
bd_df2['is_normal_flow'] = (
    bd_df2['offer received'].notna() &
    bd_df2['offer viewed'].notna() &
    bd_df2['offer completed'].notna() &
    (bd_df2['offer received'] <= bd_df2['offer viewed']) &
    (bd_df2['offer viewed'] <= bd_df2['offer completed'])
).astype(int)

# 성별 x offer_type
gender_offer_conversion = (
    bd_df2
    .groupby(['gender', 'offer_type'])
    .agg(total=('person', 'count'), normal_flow=('is_normal_flow', 'sum'))
    .reset_index()
)
gender_offer_conversion['normal_flow_rate'] = (
    gender_offer_conversion['normal_flow'] / gender_offer_conversion['total'] * 100
).round(2)
display(gender_offer_conversion)

# 성별 x offer_id
gender_offer_conversion_id = (
    bd_df2
    .groupby(['gender', 'offer_id'])
    .agg(total=('person', 'count'), normal_flow=('is_normal_flow', 'sum'))
    .reset_index()
)
gender_offer_conversion_id['normal_flow_rate'] = (
    gender_offer_conversion_id['normal_flow'] / gender_offer_conversion_id['total'] * 100
).round(2)
display(gender_offer_conversion_id)

,gender,offer_type,total,normal_flow,normal_flow_rate
0,F,bogo,11042,5253,47.57
1,F,discount,11056,5262,47.59
2,M,bogo,15295,5276,34.49
3,M,discount,15523,6380,41.10
4,O,bogo,358,191,53.35
5,O,discount,371,199,53.64


,gender,offer_id,total,normal_flow,normal_flow_rate
0,F,bogo_10_10_5,2752,1459,53.02
1,F,bogo_10_10_7,2773,1297,46.77
2,F,bogo_5_5_5,2730,1563,57.25
3,F,bogo_5_5_7,2787,934,33.51
4,F,discount_10_2_10,2719,1908,70.17
5,F,discount_10_2_7,2750,916,33.31
6,F,discount_20_5_10,2852,617,21.63
7,F,discount_7_3_7,2735,1821,66.58
8,M,bogo_10_10_5,3798,1239,32.62
9,M,bogo_10_10_7,3869,1239,32.02


In [9]:
bd_df2['age_group'] = pd.cut(bd_df2['age'], bins=[0, 20, 30, 40, 50, 60, 70, 100], labels=['10대', '20대', '30대', '40대', '50대', '60대', '70대+'])

# 연령대 x offer_type
age_offer_conversion = (
    bd_df2
    .groupby(['age_group', 'offer_type'])
    .agg(total=('person', 'count'), normal_flow=('is_normal_flow', 'sum'))
    .reset_index()
)
age_offer_conversion['normal_flow_rate'] = (
    age_offer_conversion['normal_flow'] / age_offer_conversion['total'] * 100
).round(2)
display(age_offer_conversion)

# 연령대 x offer_id
age_offer_conversion_id = (
    bd_df2
    .groupby(['age_group', 'offer_id'])
    .agg(total=('person', 'count'), normal_flow=('is_normal_flow', 'sum'))
    .reset_index()
)
age_offer_conversion_id['normal_flow_rate'] = (
    age_offer_conversion_id['normal_flow'] / age_offer_conversion_id['total'] * 100
).round(2)
display(age_offer_conversion_id)

,age_group,offer_type,total,normal_flow,normal_flow_rate
0,10대,bogo,640,158,24.69
1,10대,discount,568,193,33.98
2,20대,bogo,2451,702,28.64
3,20대,discount,2574,903,35.08
4,30대,bogo,2850,1032,36.21
5,30대,discount,2975,1240,41.68
6,40대,bogo,4337,1818,41.92
7,40대,discount,4239,1970,46.47
8,50대,bogo,6454,2763,42.81
9,50대,discount,6553,2969,45.31


,age_group,offer_id,total,normal_flow,normal_flow_rate
0,10대,bogo_10_10_5,166,37,22.29
1,10대,bogo_10_10_7,158,39,24.68
2,10대,bogo_5_5_5,169,63,37.28
3,10대,bogo_5_5_7,147,19,12.93
4,10대,discount_10_2_10,127,70,55.12
5,10대,discount_10_2_7,153,25,16.34
6,10대,discount_20_5_10,132,15,11.36
7,10대,discount_7_3_7,156,83,53.21
8,20대,bogo_10_10_5,640,150,23.44
9,20대,bogo_10_10_7,598,169,28.26


In [10]:
bd_df2['income_group'] = pd.cut(bd_df2['income'],
    bins=[0, 40000, 60000, 80000, 100000, 120000],
    labels=['4만이하', '4~6만', '6~8만', '8~10만', '10만이상'])

# 소득 구간 x offer_type
income_offer_conversion = (
    bd_df2
    .groupby(['income_group', 'offer_type'])
    .agg(total=('person', 'count'), normal_flow=('is_normal_flow', 'sum'))
    .reset_index()
)
income_offer_conversion['normal_flow_rate'] = (
    income_offer_conversion['normal_flow'] / income_offer_conversion['total'] * 100
).round(2)
display(income_offer_conversion)

# 소득 구간 x offer_id
income_offer_conversion_id = (
    bd_df2
    .groupby(['income_group', 'offer_id'])
    .agg(total=('person', 'count'), normal_flow=('is_normal_flow', 'sum'))
    .reset_index()
)
income_offer_conversion_id['normal_flow_rate'] = (
    income_offer_conversion_id['normal_flow'] / income_offer_conversion_id['total'] * 100
).round(2)
display(income_offer_conversion_id)

,income_group,offer_type,total,normal_flow,normal_flow_rate
0,4만이하,bogo,3926,995,25.34
1,4만이하,discount,3917,1330,33.95
2,4~6만,bogo,8147,2844,34.91
3,4~6만,discount,8276,3390,40.96
4,6~8만,bogo,8131,3575,43.97
5,6~8만,discount,8344,3929,47.09
6,8~10만,bogo,4647,2417,52.01
7,8~10만,discount,4581,2412,52.65
8,10만이상,bogo,1844,889,48.21
9,10만이상,discount,1832,780,42.58


,income_group,offer_id,total,normal_flow,normal_flow_rate
0,4만이하,bogo_10_10_5,947,197,20.80
1,4만이하,bogo_10_10_7,980,236,24.08
2,4만이하,bogo_5_5_5,951,373,39.22
3,4만이하,bogo_5_5_7,1048,189,18.03
4,4만이하,discount_10_2_10,978,549,56.13
5,4만이하,discount_10_2_7,956,165,17.26
6,4만이하,discount_20_5_10,1011,83,8.21
7,4만이하,discount_7_3_7,972,533,54.84
8,4~6만,bogo_10_10_5,2030,658,32.41
9,4~6만,bogo_10_10_7,2045,707,34.57


In [11]:
bd_df2['join_year'] = pd.to_datetime(bd_df2['became_member_on']).dt.year

# 가입 연도 x offer_type
join_offer_conversion = (
    bd_df2
    .groupby(['join_year', 'offer_type'])
    .agg(total=('person', 'count'), normal_flow=('is_normal_flow', 'sum'))
    .reset_index()
)
join_offer_conversion['normal_flow_rate'] = (
    join_offer_conversion['normal_flow'] / join_offer_conversion['total'] * 100
).round(2)
display(join_offer_conversion)

# 가입 연도 x offer_id
join_offer_conversion_id = (
    bd_df2
    .groupby(['join_year', 'offer_id'])
    .agg(total=('person', 'count'), normal_flow=('is_normal_flow', 'sum'))
    .reset_index()
)
join_offer_conversion_id['normal_flow_rate'] = (
    join_offer_conversion_id['normal_flow'] / join_offer_conversion_id['total'] * 100
).round(2)
display(join_offer_conversion_id)

,join_year,offer_type,total,normal_flow,normal_flow_rate
0,2013,bogo,528,163,30.87
1,2013,discount,509,264,51.87
2,2014,bogo,1230,366,29.76
3,2014,discount,1265,647,51.15
4,2015,bogo,3339,1495,44.77
5,2015,discount,3323,1655,49.80
6,2016,bogo,6302,3131,49.68
7,2016,discount,6388,3414,53.44
8,2017,bogo,11735,4221,35.97
9,2017,discount,11609,4507,38.82


,join_year,offer_id,total,normal_flow,normal_flow_rate
0,2013,bogo_10_10_5,126,31,24.60
1,2013,bogo_10_10_7,140,28,20.00
2,2013,bogo_5_5_5,121,55,45.45
3,2013,bogo_5_5_7,141,49,34.75
4,2013,discount_10_2_10,128,106,82.81
5,2013,discount_10_2_7,125,45,36.00
6,2013,discount_20_5_10,130,18,13.85
7,2013,discount_7_3_7,126,95,75.40
8,2014,bogo_10_10_5,330,78,23.64
9,2014,bogo_10_10_7,304,52,17.11
